In [20]:
import pandas as pd, numpy as np
import os

In [21]:
os.getcwd()

'/Users/jun/GitStudy/human_A/notebooks'

In [52]:
DATA_PATH = "/Users/jun/GitStudy/human_A/data/smartfarm_statefirst_2026-04-01_90d.csv"

In [53]:
df = pd.read_csv(DATA_PATH)

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129600 entries, 0 to 129599
Columns: 135 entries, timestamp to fault_history_24h_min
dtypes: float64(96), int64(32), object(7)
memory usage: 133.5+ MB


In [55]:
df.timestamp.min()

'2026-04-01 00:00:00'

In [56]:
df.timestamp.max()

'2026-06-29 23:59:00'

bearing_vibration_peak_mm_s 대신 bearing_vibration_rms_mm_s 넣어줘야 함.

In [57]:
df.describe()

,day_index,minute_of_day,hour,pump_on,fertigation_on,event_id,day_fert_start_hour,day_fert_stop_hour,events_per_day_target,weekly_flush_on,...,dp_per_flow,flow_drop_rate,flow_per_power,pressure_per_power,pressure_std_15m,pressure_iqr_15m,pressure_std_iqr_ratio,flow_std_15m,flow_cv_15m,fault_history_24h_min
count,129600.000000,129600.000000,129600.000000,129600.000000,129600.000000,129600.000000,129600.0,129600.0,129600.000000,129600.000000,...,129600.000000,129600.000000,129600.000000,129600.000000,129600.000000,129600.000000,129600.000000,129600.000000,129600.000000,129600.000000
mean,45.500000,719.500000,11.991667,0.497245,0.072824,25.431752,7.0,19.0,8.033333,0.003241,...,24.883573,0.013184,5.270832,369.097543,2.758105,4.095707,0.949918,1.683706,1.193786,26.814707
std,25.979259,415.693697,6.928228,0.499994,0.259849,109.679722,0.0,0.0,0.233334,0.056835,...,56.549394,0.139858,17.539248,382.137240,6.020813,9.967791,1.921701,4.287646,0.512032,45.552961
min,1.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,7.0,19.0,7.000000,0.000000,...,0.000000,-0.572909,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,23.000000,359.750000,5.995833,0.000000,0.000000,-1.000000,7.0,19.0,8.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.107000,0.140000,0.641638,0.007000,0.857466,0.000000
50%,45.500000,719.500000,11.991667,0.000000,0.000000,-1.000000,7.0,19.0,8.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.243000,0.300000,0.734763,0.049000,1.103979,0.000000
75%,68.000000,1079.250000,17.987500,1.000000,0.000000,-1.000000,7.0,19.0,8.000000,0.000000,...,2.263215,0.000000,0.770953,787.475164,0.620000,0.795000,0.865010,0.135000,1.393283,72.000000
max,90.000000,1439.000000,23.983333,1.000000,1.000000,722.000000,7.0,19.0,9.000000,1.000000,...,310.178998,1.000000,104.858376,2461.518307,39.679000,78.220000,138.238794,19.125000,3.872537,152.000000


In [ ]:
print(df.columns)

Index(['timestamp', 'day_index', 'minute_of_day', 'hour', 'crop_stage',
       'context_type', 'context_event', 'pump_on', 'fertigation_on',
       'event_id',
       ...
       'dp_per_flow', 'flow_drop_rate', 'flow_per_power', 'pressure_per_power',
       'pressure_std_15m', 'pressure_iqr_15m', 'pressure_std_iqr_ratio',
       'flow_std_15m', 'flow_cv_15m', 'fault_history_24h_min'],
      dtype='object', length=135)

현장의 PLC(제어반)와 센서 노드에서 직접 쏘아주는 순수 데이터들입니다. 성격별로 분류해 두면 나중에 다중공선성을 피하기 위해 그룹별로 묶을 때(Aggregation) 매우 편합니다.

1. 기본 환경 및 상태 데이터 (Environment & State)
- timestamp: 측정 시각 (가장 중요한 인덱스)

- air_temp_c, relative_humidity_pct, co2_ppm, light_ppfd_umol_m2_s: 실내 환경 센서

- pump_on, lights_on, ventilation_state, dehumidifier_state: 장비 가동 상태 (0/1)

2. 펌프 및 모터 기계 데이터 (Pump & Motor - 예지보전 핵심)

- pump_rpm: 제어기가 펌프에 내린 회전 속도 명령값

- flow_rate_l_min: 실제 토출 유량

- suction_pressure_kpa, discharge_pressure_kpa: 흡입/토출 수압

- filter_pressure_in_kpa, filter_pressure_out_kpa: 여과기 전후 수압

- motor_current_a, voltage_v, motor_power_kw: 전기적 부하 측정값

- motor_temperature_c, bearing_temperature_c: 온도 센서

- acoustic_db, bearing_vibration_rms_mm_s: 소음 및 진동 센서

- turbidity_ntu: 탁도 센서

3. 양액 조제기 제어 데이터 (Nutrient Mixer)

- raw_water_temp_c, raw_water_ec_ds_m, raw_water_ph, raw_tank_pressure_kpa, raw_tank_level_pct: 원수(지하수/수돗물) 상태

- mix_target_ec_ds_m, mix_target_ph: 농장주가 제어기에 설정한 목표값 (매우 중요)

- mix_ec_ds_m, mix_ph, mix_temp_c, mix_flow_l_min: 섞인 후 측정된 실제 센서값

- dosing_acid_ml_min: 산 주입량

- tank_a_level_pct, tank_b_level_pct, acid_tank_level_pct: 비료/산 탱크 잔량 센서

- valve_a_on, valve_b_on, valve_acid_on: 밸브 개방 여부 (0/1)

4. 구역별 베드 환경 데이터 (Zone 1~3 Substrate)

- zone1_valve_on ~ zone3_valve_on: 구역별 관수 밸브 상태

- zone1_flow_l_min ~ zone3_flow_l_min: 구역별 공급 유량

- zone1_pressure_kpa ~ zone3_pressure_kpa: 구역별 수압

- zone1_substrate_moisture_pct ~ zone3_substrate_moisture_pct: 배지 함수율(수분)

- zone1_substrate_ec_ds_m ~ zone3_substrate_ec_ds_m: 배지 EC

- zone1_substrate_ph ~ zone3_substrate_ph: 배지 pH

- drain_ec_ds_m: 모여서 버려지는 배액의 EC 센서값

In [59]:
raw_columns = [
    'timestamp', 
    'pump_on', 'lights_on', 'ventilation_state', 'dehumidifier_state',
    'air_temp_c', 'relative_humidity_pct', 'co2_ppm', 'light_ppfd_umol_m2_s',
    'pump_rpm', 'flow_rate_l_min', 'suction_pressure_kpa', 'discharge_pressure_kpa',
    'filter_pressure_in_kpa', 'filter_pressure_out_kpa', 'turbidity_ntu',
    'motor_current_a', 'voltage_v', 'motor_power_kw',
    'motor_temperature_c', 'bearing_temperature_c', 
    'acoustic_db', 'bearing_vibration_rms_mm_s',
    'raw_tank_level_pct', 'raw_water_temp_c', 'raw_water_ec_ds_m', 'raw_water_ph', 'raw_tank_pressure_kpa',
    'mix_target_ec_ds_m', 'mix_ec_ds_m', 'mix_target_ph', 'mix_ph', 'mix_temp_c', 'mix_flow_l_min',
    'tank_a_level_pct', 'tank_b_level_pct', 'acid_tank_level_pct',
    'valve_a_on', 'valve_b_on', 'valve_acid_on', 'dosing_acid_ml_min',
    'drain_ec_ds_m'
]
# Zone 1~3까지의 컬럼들을 동적으로 추가
for i in range(1, 4):
    raw_columns.extend([
        f'zone{i}_valve_on', f'zone{i}_flow_l_min', f'zone{i}_pressure_kpa',
        f'zone{i}_substrate_moisture_pct', f'zone{i}_substrate_ec_ds_m', f'zone{i}_substrate_ph'
    ])

# 기존 df에서 선별하여 새로운 df_raw 생성
# df_raw = df[raw_columns].copy()

# (필수) timestamp를 인덱스로 설정하고 시간순으로 정렬
# df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])
# df_raw = df_raw.set_index('timestamp').sort_index()

print(f"추출된 Raw 컬럼 수: {len(raw_columns)}개")

추출된 Raw 컬럼 수: 60개


In [60]:
9+14+19+18

60

In [62]:
# selected_columns // bearing_vibration_rms_mm_s 랑 bearing_vibration_peak_mm_s 랑 바꿔야 함

selected_columns = ['tank_b_level_pct',
 'zone3_substrate_ec_ds_m',
 'zone3_substrate_ph',
 'zone2_substrate_ph',
 'filter_pressure_in_kpa',
 'motor_power_kw',
 'mix_target_ec_ds_m',
 'zone3_pressure_kpa',
 'air_temp_c',
 'drainage_ratio_pct',
 'filter_pressure_out_kpa',
 'cavitation_index',
 'vibration_bandpower_high',
 'turbidity_ntu',
 'dosing_acid_ml_min',
 'zone1_substrate_moisture_pct',
 'zone1_substrate_ec_ds_m',
 'flow_rate_l_min',
 'valve_a_on',
 'raw_water_ph',
 'motor_current_a',
 'zone2_substrate_ec_ds_m',
 'timestamp',
 'flow_baseline_l_min',
 'zone2_pressure_kpa',
 'vpd_kpa',
 'drain_ec_ds_m',
 'mix_ec_ds_m',
 'valve_acid_on',
 'lights_on',
 'dehumidifier_state',
 'co2_ppm',
 'raw_water_ec_ds_m',
 'zone3_valve_on',
 'mix_target_ph',
 'discharge_pressure_kpa',
 'bearing_temperature_c',
 'zone1_flow_l_min',
 'motor_temperature_c',
 'mix_flow_l_min',
 'pump_on',
 'mix_ph',
 'mix_temp_c',
 'raw_tank_level_pct',
 'tank_a_level_pct',
 'relative_humidity_pct',
 'zone1_valve_on',
 'zone3_flow_l_min',
 'light_ppfd_umol_m2_s',
 'zone3_substrate_moisture_pct',
 'ventilation_state',
 'zone1_pressure_kpa',
 'zone2_valve_on',
 'valve_b_on',
 'zone2_flow_l_min',
 'acoustic_db',
 'zone2_substrate_moisture_pct',
 'bearing_vibration_rms_mm_s',
 'raw_tank_pressure_kpa',
 'suction_pressure_kpa',
 'voltage_v',
 'acid_tank_level_pct',
 'zone1_substrate_ph',
 'pump_rpm',
 'raw_water_temp_c']



In [63]:
len(selected_columns)

65

In [64]:
for i in selected_columns:
    if not i in raw_columns:
        print(i)

drainage_ratio_pct
cavitation_index
vibration_bandpower_high
flow_baseline_l_min
vpd_kpa


In [65]:
selected_columns = set(selected_columns)

In [66]:
selected_columns = list(selected_columns)

In [67]:
selected_columns

['mix_temp_c',
 'bearing_temperature_c',
 'acid_tank_level_pct',
 'mix_ph',
 'motor_power_kw',
 'zone2_substrate_ec_ds_m',
 'motor_temperature_c',
 'valve_acid_on',
 'acoustic_db',
 'zone1_flow_l_min',
 'zone3_substrate_ph',
 'timestamp',
 'drainage_ratio_pct',
 'zone3_pressure_kpa',
 'zone1_pressure_kpa',
 'raw_tank_pressure_kpa',
 'raw_water_ec_ds_m',
 'flow_rate_l_min',
 'zone3_substrate_ec_ds_m',
 'tank_a_level_pct',
 'filter_pressure_in_kpa',
 'zone2_substrate_ph',
 'ventilation_state',
 'zone2_flow_l_min',
 'tank_b_level_pct',
 'zone3_flow_l_min',
 'zone1_substrate_ph',
 'zone1_valve_on',
 'mix_target_ec_ds_m',
 'zone1_substrate_moisture_pct',
 'raw_tank_level_pct',
 'pump_on',
 'raw_water_temp_c',
 'valve_b_on',
 'zone3_valve_on',
 'zone3_substrate_moisture_pct',
 'valve_a_on',
 'mix_target_ph',
 'voltage_v',
 'raw_water_ph',
 'mix_ec_ds_m',
 'discharge_pressure_kpa',
 'motor_current_a',
 'light_ppfd_umol_m2_s',
 'bearing_vibration_rms_mm_s',
 'cavitation_index',
 'co2_ppm',
 'z

In [68]:
df_raw = df[selected_columns]

In [69]:
df_raw.columns

Index(['mix_temp_c', 'bearing_temperature_c', 'acid_tank_level_pct', 'mix_ph',
       'motor_power_kw', 'zone2_substrate_ec_ds_m', 'motor_temperature_c',
       'valve_acid_on', 'acoustic_db', 'zone1_flow_l_min',
       'zone3_substrate_ph', 'timestamp', 'drainage_ratio_pct',
       'zone3_pressure_kpa', 'zone1_pressure_kpa', 'raw_tank_pressure_kpa',
       'raw_water_ec_ds_m', 'flow_rate_l_min', 'zone3_substrate_ec_ds_m',
       'tank_a_level_pct', 'filter_pressure_in_kpa', 'zone2_substrate_ph',
       'ventilation_state', 'zone2_flow_l_min', 'tank_b_level_pct',
       'zone3_flow_l_min', 'zone1_substrate_ph', 'zone1_valve_on',
       'mix_target_ec_ds_m', 'zone1_substrate_moisture_pct',
       'raw_tank_level_pct', 'pump_on', 'raw_water_temp_c', 'valve_b_on',
       'zone3_valve_on', 'zone3_substrate_moisture_pct', 'valve_a_on',
       'mix_target_ph', 'voltage_v', 'raw_water_ph', 'mix_ec_ds_m',
       'discharge_pressure_kpa', 'motor_current_a', 'light_ppfd_umol_m2_s',
       'beari

In [70]:
df_raw.to_csv("/Users/jun/GitStudy/human_A/data/selected_data.csv",index=False)